In [3]:
from pymilvus import MilvusClient,DataType

In [8]:
client = MilvusClient(
    uri="http://localhost:19530",
    token="root:Milvus"
)

In [49]:
schema = client.create_schema(
    auto_id=True,
    description="My collection schema",
    enable_dynamic_field=True
)

In [50]:
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=5)
schema.add_field(field_name="color", datatype=DataType.VARCHAR, max_length=50)

{'auto_id': True, 'description': 'My collection schema', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 5}}, {'name': 'color', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 50}}], 'enable_dynamic_field': True, 'enable_namespace': False}

In [51]:
# 向量字段在集合创建的时候没要求必须创建index  但是向量字段在查询的时候需要改字段创建了index
index_paras = client.prepare_index_params()
index_paras.add_index(
    field_name="vector",
    index_type="IVF_FLAT",
    metric_type="L2"
)

In [52]:
# client.drop_collection(collection_name="my_collection")

client.create_collection(
    collection_name="my_collection", 
    schema=schema, 
    consistency_level="Strong",
    shards_num=2,
    index_params=index_paras
)

In [53]:
client.list_partitions(collection_name="my_collection")

['_default']

In [54]:
client.create_partition(collection_name="my_collection", partition_name="blue_partition")
client.create_partition(collection_name="my_collection", partition_name="red_partition")

In [55]:
client.list_partitions(collection_name="my_collection")

['_default', 'blue_partition', 'red_partition']

In [56]:
data=[
    {"id": 0, "vector": [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592], "color": "pink_8682"},
    {"id": 1, "vector": [0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104], "color": "red_7025"},
    {"id": 2, "vector": [0.43742130801983836, -0.5597502546264526, 0.6457887650909682, 0.7894058910881185, 0.20785793220625592], "color": "orange_6781"},
    {"id": 3, "vector": [0.3172005263489739, 0.9719044792798428, -0.36981146090600725, -0.4860894583077995, 0.95791889146345], "color": "pink_9298"},
    {"id": 4, "vector": [0.4452349528804562, -0.8757026943054742, 0.8220779437047674, 0.46406290649483184, 0.30337481143159106], "color": "red_4794"},
    {"id": 5, "vector": [0.985825131989184, -0.8144651566660419, 0.6299267002202009, 0.1206906911183383, -0.1446277761879955], "color": "yellow_4222"},
    {"id": 6, "vector": [0.8371977790571115, -0.015764369584852833, -0.31062937026679327, -0.562666951622192, -0.8984947637863987], "color": "red_9392"},
    {"id": 7, "vector": [-0.33445148015177995, -0.2567135004164067, 0.8987539745369246, 0.9402995886420709, 0.5378064918413052], "color": "grey_8510"},
    {"id": 8, "vector": [0.39524717779832685, 0.4000257286739164, -0.5890507376891594, -0.8650502298996872, -0.6140360785406336], "color": "white_9381"},
    {"id": 9, "vector": [0.5718280481994695, 0.24070317428066512, -0.3737913482606834, -0.06726932177492717, -0.6980531615588608], "color": "purple_4976"}
]

res = client.upsert(
    collection_name="my_collection",
    data=data
)

print(res)



{'upsert_count': 10, 'ids': [464321742241716695, 464321742241716696, 464321742241716697, 464321742241716698, 464321742241716699, 464321742241716700, 464321742241716701, 464321742241716702, 464321742241716703, 464321742241716704]}


In [66]:
# 如果id字段是auto_id=True的  就不需要提供id  只需要提供vector和color字段就可以了
data=[
    {"vector": [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592], "color": "pink_8682"},
    {"vector": [0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104], "color": "red_7025"},
    {"vector": [0.43742130801983836, -0.5597502546264526, 0.6457887650909682, 0.7894058910881185, 0.20785793220625592], "color": "orange_6781"},
    {"vector": [0.3172005263489739, 0.9719044792798428, -0.36981146090600725, -0.4860894583077995, 0.95791889146345], "color": "pink_9298"},
    {"vector": [0.4452349528804562, -0.8757026943054742, 0.8220779437047674, 0.46406290649483184, 0.30337481143159106], "color": "red_4794"},
    {"vector": [0.985825131989184, -0.8144651566660419, 0.6299267002202009, 0.1206906911183383, -0.1446277761879955], "color": "yellow_4222"},
    {"vector": [0.8371977790571115, -0.015764369584852833, -0.31062937026679327, -0.562666951622192, -0.8984947637863987], "color": "red_9392"},
    {"vector": [-0.33445148015177995, -0.2567135004164067, 0.8987539745369246, 0.9402995886420709, 0.5378064918413052], "color": "grey_8510"},
    {"vector": [0.39524717779832685, 0.4000257286739164, -0.5890507376891594, -0.8650502298996872, -0.6140360785406336], "color": "white_9381"},
    {"vector": [0.5718280481994695, 0.24070317428066512, -0.3737913482606834, -0.06726932177492717, -0.6980531615588608], "color": "purple_4976"}
]

res = client.insert(
    collection_name="my_collection",
    # highlight-next-line
    partition_name="red_partition",
    data=data
)

print(res)



{'insert_count': 10, 'ids': [464321742241716737, 464321742241716738, 464321742241716739, 464321742241716740, 464321742241716741, 464321742241716742, 464321742241716743, 464321742241716744, 464321742241716745, 464321742241716746]}


In [67]:
client.get_partition_stats(collection_name="my_collection", partition_name="red_partition")

{'row_count': 0}

In [68]:
from numpy import partition


client.load_collection(collection_name="my_collection", timeout=30)
client.get_collection_stats(collection_name="my_collection",partition_name="red_partition")

{'row_count': 10}

In [6]:
from pymilvus import CollectionSchema, FieldSchema
from regex import F


a = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True)
b = FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=5)
c = FieldSchema(name="color", dtype=DataType.VARCHAR, max_length=50)

CollectionSchema(fields=[a, b, c])

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 5}}, {'name': 'color', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 50}}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [14]:
client.get_collection_stats(collection_name="my_collection")

{'row_count': 40}

In [13]:
client.get_partition_stats(collection_name="my_collection", partition_name="red_partition")

{'row_count': 30}

In [9]:
# 分页参数
page_size = 2  # 每页查询条数
offset = 0        # 起始偏移量
all_data = []

# 循环分页查询
while True:
    # 分页查数据
    page_data = client.query(
        collection_name="my_collection",
        filter="",
        output_fields=["id", "color"],
        limit=page_size,
        offset=offset  # 偏移量，实现分页
    )
    
    if not page_data:  # 无数据时退出循环
        break
    
    all_data.extend(page_data)
    offset += page_size  # 偏移量递增
    print(f"已查询 {len(all_data)} 条数据...")

print(f"\n全量查询完成，总条数：{len(all_data)}")

已查询 2 条数据...
已查询 4 条数据...
已查询 6 条数据...
已查询 8 条数据...
已查询 10 条数据...
已查询 12 条数据...
已查询 14 条数据...
已查询 16 条数据...
已查询 18 条数据...
已查询 20 条数据...
已查询 22 条数据...
已查询 24 条数据...
已查询 26 条数据...
已查询 28 条数据...
已查询 30 条数据...
已查询 32 条数据...
已查询 34 条数据...
已查询 36 条数据...
已查询 38 条数据...
已查询 40 条数据...

全量查询完成，总条数：40


In [16]:
from sympy import limit


client.query(
    collection_name="my_collection",
    filter="",
    limit=100

)

data: ["{'id': 464321742241716695, 'vector': [0.35803765058517456, -0.602349579334259, 0.1841401308774948, -0.26286205649375916, 0.9029438495635986], 'color': 'pink_8682'}", "{'id': 464321742241716696, 'vector': [0.19886812567710876, 0.060235604643821716, 0.697696328163147, 0.2614474594593048, 0.8387295007705688], 'color': 'red_7025'}", "{'id': 464321742241716697, 'vector': [0.4374213218688965, -0.5597502589225769, 0.6457887887954712, 0.789405882358551, 0.20785793662071228], 'color': 'orange_6781'}", "{'id': 464321742241716698, 'vector': [0.31720051169395447, 0.971904456615448, -0.369811475276947, -0.48608946800231934, 0.9579188823699951], 'color': 'pink_9298'}", "{'id': 464321742241716699, 'vector': [0.4452349543571472, -0.8757026791572571, 0.8220779299736023, 0.46406289935112, 0.3033747971057892], 'color': 'red_4794'}", "{'id': 464321742241716700, 'vector': [0.9858251214027405, -0.8144651651382446, 0.6299266815185547, 0.12069068849086761, -0.14462777972221375], 'color': 'yellow_4222'

In [ ]:
#主键是一定需要指定的

data=[
    {
        "id": 1,
        "issue": "vol.14"
    },
    {
        "id": 2, 
        "issue": "vol.7"
    }
]

res = client.upsert(
    collection_name="my_collection",
    data=data,
    partial_update=True
)

print(res)



In [17]:
# 分页参数
page_size = 2  # 每页查询条数
offset = 0        # 起始偏移量
all_data = []

# 循环分页查询
while True:
    # 分页查数据
    page_data = client.query(
        collection_name="my_collection",
        filter="",
        output_fields=["id", "color"],
        limit=page_size,
        offset=offset  # 偏移量，实现分页
    )
    
    if not page_data:  # 无数据时退出循环
        break
    
    all_data.extend(page_data)
    offset += page_size  # 偏移量递增
    print(f"已查询 {len(all_data)} 条数据...")

print(f"\n全量查询完成，总条数：{len(all_data)}")

已查询 2 条数据...
已查询 4 条数据...
已查询 6 条数据...
已查询 8 条数据...
已查询 10 条数据...
已查询 12 条数据...
已查询 14 条数据...
已查询 16 条数据...
已查询 18 条数据...
已查询 20 条数据...
已查询 22 条数据...
已查询 24 条数据...
已查询 26 条数据...
已查询 28 条数据...
已查询 30 条数据...
已查询 32 条数据...
已查询 34 条数据...
已查询 36 条数据...
已查询 38 条数据...
已查询 40 条数据...

全量查询完成，总条数：40


In [ ]:
# 按条件删除实体
client.delete(
    collection_name="my_collection",
    filter="color == 'red_7025'"
)
# 按主键删除实体
client.delete(
    collection_name="my_collection",
    filter="id in [1, 2]"
)

client.delete(
    collection_name="my_collection",
    ids=[1, 2]
)
# 从分区中删除实体
client.delete(
    collection_name="my_collection",
    partition_name="red_partition",
    ids=['1','2']
)